In [9]:
import spacy
#from spacy import displacy
#from spacy.glossary import explain

from spacy.language import Language
from spacy.tokens import Doc, Token
from itertools import count

from GraphToAbridgedRecipe import render
from Node import Node

In [ ]:

utensils       = "pan, skillet, pot, blender".split(", ")
final_utensils = "plate, bowl".split(", ")          # destinations, never the "working" vessel
NOT_FOOD       = "heat, minute, hour, second, batch, serving, juice".split(", ")
movementVerbs  = "transfer, move".split(", ")
GENVERB        = "simmer, boil".split(", ")         # passive states: cook can walk away
HARD_REMOVE    = ("piece, juice, chunk, crumble, slice, dice, leg, breast, liver, "
                  "heart, thigh, loin, mince, cut, chop, pulse").split(", ")
ADDITIVE       = ("add, stir, pour, season, sprinkle, place, toss").split(", ")
SKIP_VERBS     = ("increase, decrease, raise, lower").split(", ")
UNITS          = {"second": 1, "minute": 60, "hour": 3600}

# order = preference when the exact vessel type isn't available
LOGICAL_UTENSIL = {
    "pot":     ["pot", "skillet", "pan"],
    "skillet": ["skillet", "pan", "pot"],
    "pan":     ["pan", "skillet", "pot"],
    "plate":   ["plate", "bowl"],
    "bowl":    ["bowl", "plate"],
    "blender": ["blender"],
}

GROUP_IDS = count()

#helper funct to turn string floats to floats
#NUMMOD is still technically a string
def toNumber(text: str):
    try:
        if "/" in text:
            a, b = text.split("/")
            return float(a) / float(b)
        return float(text)
    except ValueError:
        return None


#Loop through the dependency graph to find seconds, minutes, or hours,
#rip off the actual number through the nummod modifier
def getTime(verb: Token) -> float:
    skip = set()
    for c in verb.children:
        if c.dep_ == "conj" and c.pos_ == "VERB":
            skip.update(t.i for t in c.subtree)
    for t in verb.subtree:
        if t.i in skip or t.lemma_ not in UNITS:
            continue
        nums = []
        for n in t.children:
            if n.dep_ == "nummod":
                for m in [n, *n.conjuncts]:
                    v = toNumber(m.text)
                    if v is not None:
                        nums.append(v)
        if nums:
            return max(nums) * UNITS[t.lemma_]
    return 0

#what is the verb that is modifying the state of a dobj?
def getState(verb: Token) -> str:
    for c in verb.children:
        if c.dep_ == "prep":  
            #special cases, simmer/boil often used as nouns: 
            #bring to a boil; leave on simmer
            for cc in c.children:
                if cc.dep_ == "pobj" and cc.lemma_ in GENVERB:
                    return cc.lemma_
        elif c.dep_ in ("xcomp", "advcl") and c.lemma_ in GENVERB: 
            return c.lemma_
    return verb.lemma_

#set last utensil to last utensil that isn't a final state
#use list specifically for mutability and reference semantics
def instanceOfUtensil(doc: Doc, lastUtensil: list) -> str:
    for t in doc:
        if t.lemma_ in utensils:
            return t.lemma_
    return lastUtensil[0] or "temp"

#A verb acts on an object
#But this happens in a place (cook state in/over/on a pan)
#This extracts the prepositional logic to find out where
#an action is happening
def destinationOf(verb: Token):
    for p in verb.children:
        if p.dep_ == "prep":
            for o in p.children:
                if o.dep_ == "pobj" and o.lemma_ in utensils + final_utensils:
                    return o.lemma_
    return None

#chain of compounds goes
#diced honey glazed steak
#dependency graph always returns this as with
#__ <= __ <= __ <= steak, where the verb that
#acts on the entire compound noun points to steak,
#and the "<=" signifies a "compound" relationship
def chainCompounds(tok: Token) -> str:
    parts = [chainCompounds(c) for c in tok.lefts if c.dep_ == "compound"]
    if tok.lemma_ not in HARD_REMOVE or not parts:
        parts.append(tok.lemma_)
    return " ".join(p for p in parts if p)

#find the food items that a verb is modifying
def objectsOf(verb: Token) -> list:
    names = []
    for c in verb.children:
        # in "cook the steak",steak is direct obj and child of verb cook
        if c.dep_ == "dobj":
            ingredients = [c]
        # cook with ...
        #handle cases where an item is cooked in conjunction to other items
        elif c.dep_ == "prep" and c.lemma_ in ("in", "into", "with"):
            ingredients = [o for o in c.children if o.dep_ == "pobj"]
        else:
            continue
        for ing in ingredients:
            for t in [ing, *ing.conjuncts]:
                foodname = chainCompounds(t)
                #using in gives false positives of "cook steak in pan". a pan isn't a food
                #cull some false positives!
                if foodname not in utensils + final_utensils and t.lemma_ not in NOT_FOOD:
                    names.append(foodname)
    return names

#need to handle false positives-- increase the heat, etc.. 
def handleVerb(verb: Token, i: int, steps: list, doc: Doc, lastUtensil: list, contents: dict):
    if verb.lemma_ in SKIP_VERBS:
        return
    while len(steps) <= i:
        steps.append([])

    #WHERE is the action happening
    #If there is a movement from one
    #Cooking medium to another, aptly set source
    is_move = verb.lemma_ in movementVerbs
    loc = (destinationOf(verb) if is_move else None) or instanceOfUtensil(doc, lastUtensil)
    if loc in utensils:
        lastUtensil[0] = loc
    
    #WHAT is the action being taken (verb)
    state    = getState(verb)
    
    #WHO is the action acting upon
    names    = objectsOf(verb) or list(contents.get(loc, {}))   # no object -> act on contents
    
    #WHEN is this action happening (or more accurately how long will it take?)
    duration = getTime(verb)
    active   = state not in GENVERB
    group    = next(GROUP_IDS) #grouping like ingredients

    #These factors help order actions => if an action if happening
    #In a different place, to a different set of objects, 
    #The it is likely a new root node in our graph. 
    #Most recipes I read explicitely state the WHERE and WHO.
    #If unstated, it is because context obviously supplies our answer
    #But, if both of these suddenly change, we have a new action happening!
    #If context changes again, the sentence usually specifies that :
    #Stir chicken in pot. In the meantime, cut vegetable. Stir chicken again.
    #Allows us to rebuild those connections and impose a 
    #temporal metric on this vector space.

    for foodname in names:
        src = None
        if is_move:
            for u, c in contents.items():
                if foodname in c:
                    src = u
                    c.pop(foodname)
        node = Node(foodname, loc, i, state, duration, active)
        node.src, node.group = src, group
        steps[i].append(node)
        contents.setdefault(loc, {})[foodname] = True

    #handle conjunctive verbs:
    #cook AND stir implies sequentiality-- you do NOT do these at the same time step
    for c in verb.children:                                    
        if c.dep_ == "conj" and c.pos_ == "VERB":
            then = any(a.lemma_ == "then" for a in c.children)
            handleVerb(c, i + 1 if then else i, steps, doc, lastUtensil, contents)


def link(steps: list):
    nodes = [n for step in steps for n in step]
    #frontier[place] = nodes of the latest action group 
    #last_obj = node that was last operated on
    last_obj, frontier = {}, {}        
    for n in nodes:
        preds = []
        #in reference mechanics, if two consequtive steps 
        #reference the same object but not necessarily the 
        #same location (pot) it's still likely that the location
        #is the same and the step is sequental:
        #cut the beef. season the beef. (sequential)

        #handle object sequentiality 
        if n.obj in last_obj:                                    
            preds.append(last_obj[n.obj])
        if n.state not in ADDITIVE: #additive states affect EVERYTHING in a pot                    
            for place in (n.loc, n.src):
                if place and place != "temp" and place in frontier \
                        and frontier[place][0].group != n.group:
                    preds += frontier[place]
        for p in dict.fromkeys(preds):
            if p is not n:
                n.preds.append(p)
                p.succs.append(n)
        last_obj[n.obj] = n

        #handle loc sequentiality
        for place in filter(None, (n.loc, n.src)):
            if place in frontier and frontier[place][0].group == n.group:
                frontier[place].append(n)
            else:
                frontier[place] = [n]
    return [n for n in nodes if not n.preds], nodes

#
def assignVessels(nodes: list[Node], available: dict, compat: dict = LOGICAL_UTENSIL) -> dict:
    #first and last node touching v
    #these bind a subprocess in a recipe
    #ie if a recipe needs a pot to cook a potato and THEN an egg,
    #one subprocess is egg cook, the other is potato cook
    #these are independent of each other. 
    first, last = {}, {}
    for idx, n in enumerate(nodes):
        for v in (n.loc, n.src):
            if v and v != "temp":
                first.setdefault(v, idx)
                last[v] = idx

    #how many utensils do i actually have? 
    physical   = [f"{t}#{k}" for t, cnt in available.items() for k in range(cnt)]
    ptype      = {p: p.split("#")[0] for p in physical}
    busy_until = {p: -1 for p in physical}
    owner_last, used, mapping = {}, set(), {}

    for v in sorted(first, key=first.get):
        choice = None #which utensil to use
        for t in compat.get(v, [v]):
            free = [p for p in physical if ptype[p] == t and busy_until[p] < first[v]]
            free.sort(key=(lambda p: p not in used))              
            if free:
                choice = free[0]
                break
        if choice is None:
            raise ValueError(f"No available vessel for '{v}' (needed during steps "
                             f"{first[v]}-{last[v]}). Available: {available}")
        mapping[v] = choice
        used.add(choice)
        if choice in owner_last:
            a, b = owner_last[choice], nodes[first[v]]
            if b not in a.succs:
                a.succs.append(b)
                b.preds.append(a)
            b.washBefore = True
        busy_until[choice] = last[v]
        owner_last[choice] = nodes[last[v]]

    for n in nodes:
        n.loc = mapping.get(n.loc, n.loc)
        if n.src:
            n.src = mapping.get(n.src, n.src)
    return mapping


def finishTimes(nodes: list):
    done = {}
    for n in nodes:                                              
        start = max((done[p] for p in n.preds), default=0)
        done[n] = start + n.duration
    return max(done.values(), default=0), done

def makeGraph(steps: list, nlp: Language, available: dict | None = None):
    foods, contents, lastUtensil = [], {}, [""]
    for step in steps:
        for sent in nlp(step).sents:
            for clause in sent.text.split(";"):
                clause = clause.strip().rstrip(".").lower()
                if not clause:
                    continue
                doc = nlp("You " + clause)
                roots = [t for t in doc if t.dep_ == "ROOT" and t.pos_ == "VERB"] or [doc[1]]
                i = len(foods)
                for r in roots:
                    handleVerb(r, i, foods, doc, lastUtensil, contents)

    roots, nodes = link(foods)
    if available:
        assignVessels(nodes, available)
        roots = [n for n in nodes if not n.preds]                # reuse edges can remove roots
    return roots, nodes

    

In [ ]:
nlp = spacy.load("en_core_web_sm")        # try en_core_web_trf if parses are poor
recipe = [
    "Season pork chunks generously with salt and pepper; toss to distribute seasonings evenly.",
    "Heat vegetable oil in a large pot over high heat.",
    "Cook brown pork in batches so meat isn't crowded, about 7 minutes per batch.",
    "Transfer pork to a plate.",
    "Place onions into the same pot; cook and stir until translucent, 3 or 4 minutes.",
    "Add garlic and cook until fragrant, about 1 minute.",
]
roots, nodes = makeGraph(recipe, nlp, available={"pot": 1, "plate": 1})


print(render(nodes))

print("critical path (s):", finishTimes(nodes)[0])

[salt:chunk:temp:0, pepper:chunk:temp:0, salt:toss:temp:1, pepper:toss:temp:1, vegetable oil:heat:pot#0:2, pork:cook:pot#0:3, pork:transfer:plate#0:4, onion:place:pot#0:5, vegetable oil:cook:pot#0:6, onion:cook:pot#0:6, vegetable oil:stir:pot#0:6, onion:stir:pot#0:6, garlic:add:pot#0:7, vegetable oil:cook:pot#0:7, onion:cook:pot#0:7, garlic:cook:pot#0:7]
1. Chunk salt and pepper.
2. Toss salt and pepper.
3. Meanwhile, heat vegetable oil in the pot.
4. Cook pork in the pot for 7 minutes.
5. Transfer pork from the pot to the plate.
6. Meanwhile, place onion in the pot.
7. Cook vegetable oil and onion in the pot for 4 minutes.
8. Stir vegetable oil and onion in the pot.
9. Meanwhile, add garlic in the pot for 1 minute.
10. Cook vegetable oil, onion, and garlic in the pot.
critical path (s): 420.0
